In [ ]:
#| default_exp sync

# sync

> Propagate small changes in the library back to notebooks
- order: 6

The library is primarily developed in notebooks so any big changes should be made there. But sometimes, it's easier to fix small bugs or typos in the modules directly. `nbdev-update` is the command that will propagate those changes back to the corresponding notebooks.

Note that you can't create new cells or reorder cells with that functionality, so your corrections should remain limited. Exported `.py` files must contain cell IDs (format `# %% path/nb.ipynb #cell_id`); if your exports don't have IDs, run `nbdev-export` to regenerate them.

In [ ]:
#| export
from nbdev.imports import *
from nbdev.config import *
from nbdev.maker import *
from nbdev.process import *
from nbdev.export import *
from nbdev.doclinks import _iter_py_cells

from fastcore.nbio import *
from fastcore.script import *
from fastcore.xtras import *

import ast
import importlib

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
def write_nb(nb, path):
    "Write `nb` to `path` with UTF-8 encoding and LF line endings"
    new = nb2str(nb)
    path = Path(path).expanduser()
    old = path.read_text(encoding='utf-8') if path.exists() else None
    if new != old: path.write_text(new, encoding='utf-8', newline='\n')

def absolute_import(name, fname, level):
    "Unwarps a relative import in `name` according to `fname`"
    if not level: return name
    mods = list(Path(fname).parts)
    if not name: return '.'.join(mods)
    return '.'.join(mods[:len(mods)-level+1]) + f".{name}"

In [ ]:
#| hide
import tempfile
with tempfile.TemporaryDirectory() as d:
    p = Path(d)/'unicode notebook.ipynb'
    write_nb(new_nb([mk_cell('π = 3.14')]), p)
    raw = p.read_bytes()
    assert bytes([13, 10]) not in raw
    test_eq(read_nb(p).cells[0].source, 'π = 3.14')

In [ ]:
#| hide
def _mode_arg(call, pos):
    mode = next((k.value for k in call.keywords if k.arg == 'mode'), None)
    if mode is None and len(call.args) > pos: mode = call.args[pos]
    if mode is None: return 'r'
    return mode.value if isinstance(mode, ast.Constant) and isinstance(mode.value, str) else None

def _text_io_issues(path, root):
    tree = ast.parse(path.read_text(encoding='utf-8'))
    for call in (o for o in ast.walk(tree) if isinstance(o, ast.Call)):
        attr = call.func.attr if isinstance(call.func, ast.Attribute) else None
        name = call.func.id if isinstance(call.func, ast.Name) else None
        kws = {k.arg for k in call.keywords}
        if attr == 'read_text' and 'encoding' not in kws:
            yield f'{path.relative_to(root)}:{call.lineno}: read_text needs encoding'
        elif attr == 'write_text':
            if 'encoding' not in kws: yield f'{path.relative_to(root)}:{call.lineno}: write_text needs encoding'
            if 'newline' not in kws: yield f'{path.relative_to(root)}:{call.lineno}: write_text needs newline'
        elif attr == 'open' or name == 'open':
            mode = _mode_arg(call, 1 if name == 'open' else 0)
            if mode is None or 'b' in mode or ':' in mode: continue
            if 'encoding' not in kws: yield f'{path.relative_to(root)}:{call.lineno}: open needs encoding'
            if any(c in mode for c in 'wax+') and 'newline' not in kws:
                yield f'{path.relative_to(root)}:{call.lineno}: open needs newline'

root = get_config().config_path
issues = [issue for path in (root/'nbdev').rglob('*.py') for issue in _text_io_issues(path, root)]
test_eq(issues, [])

In [ ]:
test_eq(absolute_import('xyz', 'nbdev', 0), 'xyz')
test_eq(absolute_import('', 'nbdev', 1), 'nbdev')
test_eq(absolute_import(None, 'nbdev', 1), 'nbdev')
test_eq(absolute_import('core', 'nbdev', 1), 'nbdev.core')
test_eq(absolute_import('core', 'nbdev/vision', 2), 'nbdev.core')  # from ..core import *
test_eq(absolute_import('transform', 'nbdev/vision', 1), 'nbdev.vision.transform')  # from .transform import *
test_eq(absolute_import('notebook.core', 'nbdev/data', 2), 'nbdev.notebook.core')  # from ..notebook.core import *

In [ ]:
#| export
@functools.lru_cache(maxsize=None)
def _mod_files():
    midx_spec = importlib.util.spec_from_file_location("_modidx", get_config().lib_path / "_modidx.py")
    midx = importlib.util.module_from_spec(midx_spec)
    midx_spec.loader.exec_module(midx)

    return L(files for mod in midx.d['syms'].values() for _,files in mod.values()).unique()

In [ ]:
#| export
_re_import = re.compile(r"from\s+\S+\s+import\s+\S")

In [ ]:
#| hide
assert _re_import.match('from foo import bar')
assert not _re_import.match('#from foo import bar')

In [ ]:
#| export
def _to_absolute(code, py_path, lib_dir):
    if not _re_import.search(code): return code
    res = update_import(code, ast.parse(code).body, str(py_path.relative_to(lib_dir).parent), absolute_import)
    return ''.join(res) if res else code

In [ ]:
#| export
def _update_nb(nb_path, cells, lib_dir):
    "Update notebook `nb_path` with contents from `cells`"
    nbp = NBProcessor(nb_path, ExportModuleProc(), rm_directives=False)
    nbp.process()
    # Build a dict of cell_id -> cell for fast lookup
    nb_cells_by_id = {c['id']: c for c in nbp.nb.cells if c.get('id')}
    for cell in cells:
        assert cell.nb_path == nb_path
        nbcell = nb_cells_by_id.get(cell.cell_id)
        if nbcell is None:
            raise ValueError(f"Cell ID '{cell.cell_id}' not found in notebook '{nb_path}'")
        dirs,_ = nbcell._partition()
        nbcell.source = ''.join(dirs) + _to_absolute(cell.code, cell.py_path, lib_dir)
    write_nb(nbp.nb, nb_path)

In [ ]:
#| export
def _update_mod(py_path, lib_dir):
    "Propagate changes from cells in module `py_path` to corresponding notebooks"
    py_cells = L(_iter_py_cells(py_path)).filter(lambda o: o.nb != 'auto')
    for nb_path,cells in groupby(py_cells, 'nb_path').items(): _update_nb(nb_path, cells, lib_dir)

In [ ]:
#| hide
assert min(_mod_files().map(lambda x: x.endswith('.py'))) is True

In [ ]:
#| export
@call_parse
def nbdev_update(fname:str=None): # A Python file name to update
    "Propagate change in modules matching `fname` to notebooks that created them"
    if fname and fname.endswith('.ipynb'): raise ValueError("`nbdev-update` operates on .py files.  If you wish to convert notebooks instead, see `nbdev-export`.")
    if os.environ.get('IN_TEST',0): return
    cfg = get_config()
    fname = Path(fname or cfg.lib_path)
    lib_dir = cfg.lib_path.parent
    files = globtastic(fname, file_glob='*.py', skip_folder_re='^[_.]').filter(lambda x: str(Path(x).absolute().relative_to(lib_dir) in _mod_files()))
    files.map(_update_mod, lib_dir=lib_dir)

In [ ]:
#| hide
# nbdev_update("../nbdev/sync.py")

## Export -

In [ ]:
#| hide
#| eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()